In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
NMPC — Simulación Parámetros Planta Real
=========================================
Migrado desde MATLAB a Python para uniformidad con el resto de códigos.

Parámetros físicos calibrados experimentalmente:
  - Atank = 3.02e-3 m²  (tanque PVC cilíndrico D=6.2cm)
  - αo = 0.0271, Do = 10mm  (calibrado por vaciado libre)
  - qi1max=2.69e-5, qi3max=2.41e-5 m³/s  (bombas R385 al 70%)
  - Referencias: 0.10 → 0.20 → 0.15 m
  - σ = 0 mm  (simulación con estado completo, sin observador)
  - Estado máximo: 0.25 m (h1_ss ≈ 0.221 m a h2=0.20 m)

Comparar RMSE_SS con KoopmanMPC para tabla de control puro (Tabla 5 tesis).
"""

import numpy as np
from scipy.optimize import least_squares
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')
output_dir = Path('./outputs_sim')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS — PLANTA REAL CALIBRADA
# ============================================================================
P = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,    'eta'      : 8.9e-4,   'g'        : 9.81,
    'alphao1' : 0.0271,   'Do1'      : 10e-3,
    'alphao2' : 0.0271,   'A2'       : 7.85e-5,
    'alphao3' : 0.0271,   'Do3'      : 10e-3,
    'alpha120': 0.3038,   'D12'      : 10e-3,
    'A12'     : 7.85e-5,  'lambdac12': 24000,
    'alpha230': 0.1344,   'D23'      : 10e-3,
    'A23'     : 7.85e-5,  'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

Ts = 2.0; T_sim = 800; Nsim = int(T_sim / Ts); SETTLING_S = 60
EPS = 1e-10; EPS_S = 1e-8

# ============================================================================
# MODELO PYTHON
# ============================================================================
def nl_ode(x, u, p=P):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh12)+EPS)
    a12  = p['alpha120']*np.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12']*np.sqrt(2*p['g']*abs(dh12)+EPS)*np.sign(dh12+1e-15)
    dh23 = h2 - h3
    l23  = p['D23']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh23)+EPS)
    a23  = p['alpha230']*np.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23']*np.sqrt(2*p['g']*abs(dh23)+EPS)*np.sign(dh23+1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4_py(x, u, p=P, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x+dt/2*k1, u, p)
    k3 = nl_ode(x+dt/2*k2, u, p); k4 = nl_ode(x+dt*k3, u, p)
    return np.clip(x + dt/6*(k1+2*k2+2*k3+k4), 0, 0.25)

def compute_ss(h2_ref, p=P):
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode([max(h1,1e-4), h2_ref, max(h3,1e-4)],
                      [np.clip(u1,0,1), 0.0], p)
    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5), (h2_ref*1.6, h2_ref*0.7, 0.7),
        (h2_ref*2.0, h2_ref*0.8, 0.9), (h2_ref*2.5, h2_ref*0.8, 0.7),
        (h2_ref*2.5, h2_ref*0.8, 0.75),(h2_ref*1.8, h2_ref*0.85, 0.85),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01,0.01,0.0],[0.25,0.25,1.0]),
                              max_nfev=5000)
            if s.success and np.linalg.norm(s.fun) < best_res:
                best_res = np.linalg.norm(s.fun); best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2],0,1), 0.0])
    return None, None

# ============================================================================
# MODELO CASADI + RK4 SIMBÓLICO
# ============================================================================
def build_casadi_model(p=P):
    nx, nu = 3, 2
    x_s = ca.SX.sym('x', nx); u_s = ca.SX.sym('u', nu)
    h1, h2, h3 = x_s[0], x_s[1], x_s[2]

    qi1 = p['qi1max'] * u_s[0]; qi3 = p['qi3max'] * u_s[1]
    h1p = (h1 + ca.sqrt(h1**2 + EPS_S)) / 2
    h2p = (h2 + ca.sqrt(h2**2 + EPS_S)) / 2
    h3p = (h3 + ca.sqrt(h3**2 + EPS_S)) / 2

    qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*ca.sqrt(2*p['g']*h1p)
    qo2 = p['alphao2']*p['A2']              *ca.sqrt(2*p['g']*h2p)
    qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*ca.sqrt(2*p['g']*h3p)

    dh12 = h1-h2; abs12 = ca.sqrt(dh12**2+EPS_S)
    l12  = p['D12']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs12)
    a12  = p['alpha120']*ca.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12']*ca.sqrt(2*p['g']*abs12)*ca.sign(dh12)

    dh23 = h2-h3; abs23 = ca.sqrt(dh23**2+EPS_S)
    l23  = p['D23']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs23)
    a23  = p['alpha230']*ca.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23']*ca.sqrt(2*p['g']*abs23)*ca.sign(dh23)

    xdot = ca.vertcat(
        (qi1-q12-qo1)/p['Atank'],
        (q12-q23-qo2)/p['Atank'],
        (qi3+q23-qo3)/p['Atank'],
    )
    f_cas = ca.Function('f', [x_s, u_s], [xdot])
    k1r = f_cas(x_s, u_s); k2r = f_cas(x_s+Ts/2*k1r, u_s)
    k3r = f_cas(x_s+Ts/2*k2r, u_s); k4r = f_cas(x_s+Ts*k3r, u_s)
    F_rk4 = ca.Function('F', [x_s, u_s], [x_s + Ts/6*(k1r+2*k2r+2*k3r+k4r)])
    return F_rk4, nx, nu

# ============================================================================
# CONSTRUCCIÓN DEL SOLVER NMPC
# ============================================================================
def build_nmpc(F_rk4, nx, nu):
    print("Construyendo NMPC...")
    N   = 20; q = 500
    R1  = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    cT  = np.array([0.0, 1.0, 0.0])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])
    xmin = np.array([0.0, 0.0, 0.0]); xmax = np.array([0.25, 0.25, 0.25])

    X_d  = ca.SX.sym('X', nx, N+1); U_d = ca.SX.sym('U', nu, N)
    x0_p = ca.SX.sym('x0', nx); yr_p = ca.SX.sym('yr'); up_p = ca.SX.sym('up', nu)

    J = 0; g = []; lbg = []; ubg = []
    g += [X_d[:,0] - x0_p]; lbg += [0.0]*nx; ubg += [0.0]*nx

    for k in range(N):
        g   += [X_d[:,k+1] - F_rk4(X_d[:,k], U_d[:,k])]
        lbg += [0.0]*nx; ubg += [0.0]*nx
        yk  = ca.dot(ca.DM(cT), X_d[:,k+1])
        ey  = yk - yr_p
        du  = U_d[:,0]-up_p if k==0 else U_d[:,k]-U_d[:,k-1]
        wk  = 10 if k == N-1 else 1
        J  += wk*q*ey**2
        J  += ca.mtimes(U_d[:,k].T, ca.DM(R1) @ U_d[:,k])
        J  += ca.mtimes(du.T, ca.DM(R2) @ du)

    w_nmpc   = ca.vertcat(ca.reshape(X_d, nx*(N+1), 1), ca.reshape(U_d, nu*N, 1))
    par_nmpc = ca.vertcat(x0_p, yr_p, up_p)
    lbw = np.concatenate([np.tile(xmin, N+1), np.tile(umin, N)])
    ubw = np.concatenate([np.tile(xmax, N+1), np.tile(umax, N)])

    nlp  = {'x': w_nmpc, 'f': J, 'g': ca.vertcat(*g), 'p': par_nmpc}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 1000,
            'ipopt.tol': 1e-5, 'print_time': 0}
    solver = ca.nlpsol('solver_nmpc', 'ipopt', nlp, opts)
    print(f"  NMPC listo (N={N}, q={q}).")
    return solver, N, umin, umax, lbw, ubw, lbg, ubg, xmin, xmax, nx, nu

def make_warm_start(x0, u0, N, xmin, xmax, nx, nu):
    Xw = np.zeros((nx, N+1)); Xw[:,0] = x0
    for i in range(N):
        Xw[:,i+1] = np.clip(rk4_py(Xw[:,i], u0), xmin, xmax)
    return np.concatenate([Xw.flatten(order='F'), np.tile(u0, N)])

# ============================================================================
# SIMULACIÓN PRINCIPAL
# ============================================================================
def main():
    print("=" * 65)
    print("  NMPC — Simulación Parámetros Planta Real")
    print("  Ref: 0.10 → 0.20 → 0.15 m | T=800s | Ts=2s")
    print("=" * 65)

    print("\nCalculando estados estacionarios...")
    SS = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)}  u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → SS NO encontrado ✗")

    F_rk4, nx, nu = build_casadi_model()
    solver, N, umin, umax, lbw, ubw, lbg, ubg, xmin, xmax, nx, nu = \
        build_nmpc(F_rk4, nx, nu)

    t1 = int(Nsim * 0.33); t2 = int(Nsim * 0.66)
    yref_v = np.concatenate([
        0.10 * np.ones(t1), 0.20 * np.ones(t2 - t1), 0.15 * np.ones(Nsim - t2),
    ])
    ref_change_times_s = [t1*Ts, t2*Ts]

    x_ss0, u_ss0 = SS[0.10]
    x0 = x_ss0.copy()
    print(f"\nCondición inicial (SS h2=0.10m): {np.round(x0,4)}")

    T_hist = np.arange(Nsim+1) * Ts
    X_hist = np.zeros((nx, Nsim+1)); X_hist[:,0] = x0
    U_hist = np.zeros((nu, Nsim)); step_ms = []

    uprev = u_ss0.copy()
    w0    = make_warm_start(x0, uprev, N, xmin, xmax, nx, nu)

    print(f"\nSimulando {Nsim} pasos...")
    t_total = time.time()

    for k in range(Nsim):
        xk = X_hist[:,k]; yr = yref_v[k]

        if k > 0 and yref_v[k] != yref_v[k-1] and yr in SS:
            x_ss_new, u_ss_new = SS[yr]
            w0 = make_warm_start(x_ss_new, u_ss_new, N, xmin, xmax, nx, nu)

        t0  = time.time()
        par = np.concatenate([xk, [yr], uprev])
        sol = solver(x0=w0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg, p=par)
        step_ms.append((time.time()-t0)*1000)

        w_opt = np.array(sol['x']).flatten()
        Xtraj = w_opt[:nx*(N+1)].reshape(N+1, nx).T
        Utraj = w_opt[nx*(N+1):].reshape(N, nu).T
        uk    = np.clip(Utraj[:,0], umin, umax)

        U_hist[:,k] = uk; uprev = uk.copy()
        X_hist[:,k+1] = rk4_py(xk, uk)

        Xsh = np.hstack([Xtraj[:,1:], Xtraj[:,-1:]])
        Ush = np.hstack([Utraj[:,1:], Utraj[:,-1:]])
        w0  = np.concatenate([Xsh.flatten(order='F'), Ush.flatten(order='F')])

        if k % 50 == 0:
            print(f"  t={k*Ts:5.0f}s | h2={X_hist[1,k+1]:.4f}m | "
                  f"ref={yr:.4f}m | u=[{uk[0]:.3f},{uk[1]:.3f}] | "
                  f"{step_ms[-1]:.1f}ms")

    total_s = time.time() - t_total
    print(f"Total: {total_s:.1f}s")

    # ── Métricas ──────────────────────────────────────────────────────────
    ref_full = np.append(yref_v, yref_v[-1])
    e_full   = (X_hist[1,:] - ref_full) * 100
    mask_400 = T_hist > 400
    settling_mask = np.ones(len(T_hist), dtype=bool)
    for tc in ref_change_times_s:
        settling_mask &= ~((T_hist >= tc) & (T_hist < tc + SETTLING_S))
    mask_ss = mask_400 & settling_mask

    rmse_400 = np.sqrt(np.mean(e_full[mask_400]**2))
    rmse_ss  = np.sqrt(np.mean(e_full[mask_ss]**2))
    bias_ss  = np.mean(e_full[mask_ss])
    mae_ss   = np.mean(np.abs(e_full[mask_ss]))
    avg_ms   = np.mean(step_ms)
    p99_ms   = np.percentile(step_ms, 99)

    print(f"\n{'='*60}")
    print(f"  MÉTRICAS NMPC — Simulación Planta Real")
    print(f"{'='*60}")
    print(f"  RMSE h2 (t>400s, incl. transitorios): {rmse_400:.4f} cm")
    print(f"  RMSE h2 (estado estacionario SS):      {rmse_ss:.4f} cm  ← comparable")
    print(f"  MAE  h2 (SS):                          {mae_ss:.4f} cm")
    print(f"  Bias h2 (SS):                          {bias_ss:.4f} cm")
    print(f"  Tiempo total simulación:               {total_s:.1f} s")
    print(f"  Tiempo medio/paso:                     {avg_ms:.1f} ms/paso")
    print(f"  Tiempo p99/paso:                       {p99_ms:.1f} ms/paso")

    # Tabla comparativa con KoopmanMPC (valores de koopman_controller.ipynb)
    kmpc_rmse_ss = 0.0089   # cm — de koopman_controller
    kmpc_avg_ms  = 1.1      # ms — de koopman_controller
    kmpc_p99_ms  = 6.4      # ms — de koopman_controller
    print(f"\n{'='*65}")
    print(f"  TABLA COMPARATIVA — NMPC vs KoopmanMPC (estado completo)")
    print(f"{'='*65}")
    print(f"  {'Métrica':<38} {'NMPC':>10} {'KoopmanMPC':>12}")
    print(f"  {'-'*62}")
    print(f"  {'RMSE ctrl h2 (SS) [cm]':<38} {rmse_ss:>10.4f} {kmpc_rmse_ss:>12.4f}")
    print(f"  {'MAE  ctrl h2 (SS) [cm]':<38} {mae_ss:>10.4f} {'—':>12}")
    print(f"  {'Bias ctrl h2 (SS) [cm]':<38} {bias_ss:>10.4f} {'—':>12}")
    print(f"  {'Tiempo medio/paso [ms]':<38} {avg_ms:>10.1f} {kmpc_avg_ms:>12.1f}")
    print(f"  {'Tiempo p99/paso [ms]':<38} {p99_ms:>10.1f} {kmpc_p99_ms:>12.1f}")

    # ── Gráficas — estilo unificado con Koopman Closed-Loop v3 ────────────
    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    fig.suptitle(
        f'NMPC — Simulación\n'
        f'RMSE_SS={rmse_ss:.3f} cm | Bias={bias_ss:.3f} cm | '
        f'media={avg_ms:.1f} ms/paso | p99={p99_ms:.1f} ms/paso',
        fontsize=9, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(T_hist, X_hist[0]*100, 'b', lw=1.5, label='$h_1$')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 1 (estado completo)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(T_hist, X_hist[1]*100, 'b', lw=2, label='$h_2$ (output)')
    ax.stairs(ref_full[:-1]*100, T_hist, color='k', linestyle='--', lw=1.5, label='Referencia')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 2 — salida controlada')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(T_hist, X_hist[2]*100, color=[0,.6,0], lw=1.5, label='$h_3$')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 3 (estado completo)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(T_hist, e_full, 'r', lw=1.5,
            label=f'RMSE_SS={rmse_ss:.3f} cm | Bias={bias_ss:.3f} cm')
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(ref_change_times_s):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error de seguimiento')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 0]
    ax.stairs(U_hist[0], T_hist, color='b', lw=1.5, label='$u_1$ (Bomba 1)')
    ax.stairs(U_hist[1], T_hist, color='r', lw=1.5, label='$u_3$ (Bomba 2)')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]'); ax.set_xlabel('t [s]')
    ax.set_title('Señales de control')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.plot(step_ms, 'k', lw=0.8, alpha=0.7)
    ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Media={avg_ms:.1f} ms')
    ax.axhline(p99_ms, color='r', lw=1.5, ls=':',  label=f'P99={p99_ms:.1f} ms')
    ax.set_ylabel('Tiempo NMPC [ms]'); ax.set_xlabel('Paso k')
    ax.set_title('Tiempo de cómputo por paso')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = output_dir / 'nmpc_planta_real.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
    print(f"\nFigura guardada: {fname}")

if __name__ == '__main__':
    main()

  NMPC — Simulación Parámetros Planta Real
  Ref: 0.10 → 0.20 → 0.15 m | T=800s | Ts=2s

Calculando estados estacionarios...
  h2=0.10m → x_ss=[0.114  0.1    0.0816]  u_ss=[0.3293 0.    ] ✓
  h2=0.15m → x_ss=[0.1676 0.15   0.1267]  u_ss=[0.4043 0.    ] ✓
  h2=0.20m → SS NO encontrado ✗
Construyendo NMPC...
  NMPC listo (N=20, q=500).

Condición inicial (SS h2=0.10m): [0.114  0.1    0.0816]

Simulando 400 pasos...
  t=    0s | h2=0.1000m | ref=0.1000m | u=[0.317,0.022] | 10.2ms
  t=  100s | h2=0.0999m | ref=0.1000m | u=[0.309,0.024] | 8.5ms
  t=  200s | h2=0.0999m | ref=0.1000m | u=[0.309,0.024] | 9.5ms
  t=  300s | h2=0.1973m | ref=0.2000m | u=[0.531,0.036] | 7.0ms
  t=  400s | h2=0.1998m | ref=0.2000m | u=[0.439,0.034] | 6.5ms
  t=  500s | h2=0.1998m | ref=0.2000m | u=[0.438,0.034] | 7.0ms
  t=  600s | h2=0.1499m | ref=0.1500m | u=[0.370,0.029] | 13.0ms
  t=  700s | h2=0.1498m | ref=0.1500m | u=[0.379,0.029] | 9.0ms
Total: 3.6s

  MÉTRICAS NMPC — Simulación Planta Real
  RMSE h2 (t>40